In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .getOrCreate()
    )
    return spark


trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
parquet_root = trec_root / "data/enwiki/parquet"
spark = get_spark()

page = spark.read.parquet((parquet_root / "page").as_posix())
nodes = spark.read.parquet((dataset_root / "graph/v2/bge-m3-knn/nodes").as_posix())
pagerank = spark.read.parquet(
    (dataset_root / "centrality/v2/bge-m3-knn/pagerank.parquet").as_posix()
)
# hits = spark.read.parquet((dataset_root / "centrality/v2/bge-m3-knn/hits.parquet").as_posix())
degree_centrality = spark.read.parquet(
    (dataset_root / "centrality/v2/bge-m3-knn/degree_centrality.parquet").as_posix()
)
degree = spark.read.parquet(
    (dataset_root / "centrality/v2/bge-m3-knn/degree.parquet").as_posix()
)

In [ ]:
df = (
    nodes.join(
        page.select(F.col("page_id").alias("id"), F.col("page_title").alias("title")),
        on="id",
        how="inner",
    )
    .join(pagerank, on="id", how="inner")
    # .join(hits, on="id", how="inner")
    .join(degree_centrality, on="id", how="inner")
    .join(degree, on="id", how="inner")
)
df.printSchema()
df.show()

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- pagerank: double (nullable = true)
 |-- in_degree_centrality: double (nullable = true)
 |-- out_degree_centrality: double (nullable = true)
 |-- in_degree: long (nullable = true)
 |-- out_degree: long (nullable = true)



+----+--------------------+--------------------+--------------------+---------------------+---------+----------+
|  id|               title|            pagerank|in_degree_centrality|out_degree_centrality|in_degree|out_degree|
+----+--------------------+--------------------+--------------------+---------------------+---------+----------+
| 705|  Politics_of_Angola|1.511891327670393E-7|3.371518170441885E-5| 7.408268625634635E-6|      223|        49|
|1010|            April_15|1.511891327670393E-7|8.768971026261405E-6| 7.408268625634635E-6|       58|        49|
|1175|             April_1|1.511891327670393E-7|7.559457781259832E-6| 7.408268625634635E-6|       50|        49|
|1202|              Applet|1.511891327670393E-7|8.164214403760618E-6| 7.408268625634635E-6|       54|        49|
|1217|            Anguilla|1.511891327670393E-7|1.406059147314328...| 7.408268625634635E-6|       93|        49|
|1360|           Anazarbus|1.511891327670393E-7|5.442809602507079E-6| 7.408268625634635E-6|     

In [13]:
df.describe().show()

25/07/24 03:13:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+-------+--------------------+--------------------+---------------------+------------------+----------+
|summary|                 id|  title|            pagerank|in_degree_centrality|out_degree_centrality|         in_degree|out_degree|
+-------+-------------------+-------+--------------------+--------------------+---------------------+------------------+----------+
|  count|            6596109|6596109|             6596109|             6596109|              6596109|           6596109|   6596109|
|   mean|3.310663828711351E7|    NaN|1.511891327664094...|7.414053853212014E-6| 7.408268625646733E-6|49.038264831584804|      49.0|
| stddev|2.305902528093172E7|    NaN|                 0.0|6.558250126854208E-6| 9.473268288880855...| 43.37778129479301|       0.0|
|    min|                 12|     !!|1.511891327670393E-7|                 0.0| 7.408268625634635E-6|                 0|        49|
|    max|           77104121|     𝼝|1.511891327670393E-7|2.792463704397382E-

In [15]:
df.where("in_degree = 0").count()

14219

In [16]:
df.where("in_degree = 0").select("id", "title").sample(0.1).show(truncate=False)

+--------+----------------------------------------+
|id      |title                                   |
+--------+----------------------------------------+
|1497021 |Nikhila_Orissa_Beedi_Shramika_Federation|
|1986147 |Angela_Kincaid                          |
|6284990 |Rafail's_Cross                          |
|9611783 |Michel_Murat                            |
|12163728|Totum_duplex                            |
|12595042|A_Few_More_Published_Studies            |
|16078096|Kiruma                                  |
|22629539|Liberator_Shapes                        |
|23402447|Got_Nuffin                              |
|23573162|Úmonín                                  |
|25588261|Winvian                                 |
|26231232|Marcin_Niewalda                         |
|31152610|Atila_Huseyin                           |
|33052413|Malik_Ata_Muhammad_Khan                 |
|36806800|Phyllogonostreptus_nigrolabiatus        |
|38925585|Carpenter_fish                          |
|39195421|Pi